In [1]:
from nltk.corpus import movie_reviews

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import re

In [ ]:
# 사용자 정의 토크나이져
class SimpleTokenizer:
    def __init__(self,num_words = 10000, oov_token='UNK'):
        self.num_words = num_words
        self.oov_token = oov_token
        self.word_index = {}
        self.index_word = {}

    def _clean_text(self,text):
        return re.findall(r'\w+',text.lower())  # re.sub(r'[^\w\s]','',temp).strip()
    
    def fit_on_texts(self,texts):
        '''빈도순으로 상위 단어 추출 토큰을 숫자로 변경 공백을 기준으로 토큰분류'''
        word_counts = Counter()
        for text in texts:
            word_counts.update(self._clean_text(text))
        # 빈도순으로 num_words 단어 추출
        most_common = word_counts.most_common(self.num_words-2)  # pad unk 특수토큰 자리 남겨
        # 0:padding 1: oov(out of vocaburay)
        self.word_index = {self.oov_token : 1}
        for i, (word,_) in enumerate(most_common):
            self.word_index[word] = i + 2
        self.index_word = {idx : w for w, idx in self.word_index.items()}
    def texts_to_sequence(self, texts):
        sequence = []
        for text in texts:
            seq = []
            for word in self._clean_text(text):
                seq.append(self.word_index.get(word,1))
            sequence.append(seq)
        return sequence

def pad_sequence(seqeuces, maxlen ,padding='pre', truncating='pre'):
    features = np.zeros(len(seqeuces), maxlen, dtype=int)
    for i,seq in enumerate(seqeuces):
        if len(seq) > maxlen:
            if truncating == 'pre':
                features[i, :] = seq[-maxlen:]
            else:
                features[i, :] = seq[:maxlen]
        else:
            if padding == 'pre':
                features[i, -len(seq):] = seq
            else:
                features[i, :len(seq)] = seq
    return features

In [ ]:
# 리뷰데이터로 적용해서 오류없는지 확인 및 수정